# Select Monocyte Runs for Masked Counterfactual Evaluation.

## Prepare the Notebook.

### Import the Libraries.

In [1]:
import joblib
import os
import sys
import yaml

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import gaussian_kde
import seaborn as sns
from tqdm import tqdm

### Read Annotation Configurations from Loop 3.

In [3]:
config3_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/annotation/bloodplus_loop3.yaml"
with open(config3_path, "r") as fb:
    annot3_dict = yaml.safe_load(fb)
protocol_cols3 = annot3_dict["protocol_columns"]
resc_protocol_cols3 = [f"{c}:rescaled" for c in protocol_cols3]

### Read Constraints Configurations from Loop 3.

In [5]:
constraints3_config_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/constraints/default_all_cols_loop3.yaml"
with open(constraints3_config_path, "r") as fb:
    constraints3_config = yaml.safe_load(fb)
ubound3_dict = constraints3_config["ubound_dict"]
lbound3_dict = constraints3_config["lbound_dict"]


## Read Optimization Results.

### Define Function to Read Results Directory in a Parallelized Way.

In [ ]:
def get_opt_df(
    res_base_dir
):
    all_runs = []

    opt_modes_list = os.listdir(res_base_dir)

    for opt_leaf_dir in opt_modes_list:
        opt_dir = os.path.join(res_base_dir, opt_leaf_dir)
        cts_list = os.listdir(opt_dir)

        for ct_leaf_dir in cts_list:
            ct_dir = os.path.join(opt_dir, ct_leaf_dir)
            runs_list = os.listdir(ct_dir)

            for run_leaf_dir in runs_list:
                run_dir = os.path.join(ct_dir, run_leaf_dir)
                csv_path = os.path.join(run_dir, "candidates.csv")            
                all_runs.append(
                    {
                        "target_ct": ct_leaf_dir,
                        "run_id": run_leaf_dir,
                        "opt_id": opt_leaf_dir,
                        "csv_path": csv_path,
                    }
                )

    def _load_run(run_dict):
        target_ct = run_dict["target_ct"]
        run_id = run_dict["run_id"]
        opt_id = run_dict["opt_id"]
        csv_path = run_dict["csv_path"]

        df = pd.read_csv(csv_path)
        df["run_id"] = run_id
        df["opt_id"] = opt_id
        df["target_ct"] = target_ct
        return df
        
    n_jobs = 100
    data = joblib.Parallel(n_jobs=n_jobs)(joblib.delayed(_load_run)(run_dict) for run_dict in all_runs)
    return pd.concat(data, ignore_index=True)

### Read Optimization Data from Looop 3.

In [8]:
res_base_dir3 = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/loop3_replicate"
opt_df3 = get_opt_df(res_base_dir3)
opt_df3

,sample_id,gm-csf_[ng_ml],tpo_[ng_ml],sr1_[nm],um171_[nm],um729_[µm],scf_[ng_ml],butyzamide_[nm],retinoic_acid_[µm],ldl_[ng_ml],...,cfg:constraints:ubound_dict:um729_[µm],cfg:constraints:ubound_dict:sr1_[nm],cfg:constraints:ubound_dict:scf_[ng_ml],cfg:constraints:ubound_dict:butyzamide_[nm],cfg:constraints:ubound_dict:retinoic_acid_[µm],cfg:constraints:ubound_dict:ldl_[ng_ml],cfg:constraints:ubound_dict:il3_[ng_ml],cfg:constraints:ubound_dict:il7_[ng_ml],cfg:constraints:ubound_dict:mcsf_[ng_ml],cfg:constraints:ubound_dict:ly_cocktail_[ul/well]
0,CD16Mono:2026-06-26_23-29-58_dfbf9997:0,4.502639,0.184500,4.242442,0.197834,1.214640,0.487637,8.061746,0.179712,0.006153,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CD16Mono:2026-06-26_23-29-58_dfbf9997:1,10.106312,2.453917,738.088200,1430.027600,0.000000,49.336456,0.088904,0.131064,0.122932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CD16Mono:2026-06-26_23-29-58_dfbf9997:2,10.268727,2.255093,66.239860,0.060196,0.091318,47.117905,0.074269,0.623657,0.054582,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CD16Mono:2026-06-26_23-29-58_dfbf9997:3,9.620847,2.409060,259.914370,957.807860,0.088218,55.726196,0.377347,0.000000,11.627491,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CD16Mono:2026-06-26_23-29-58_dfbf9997:4,9.052049,2.468425,1285.206400,1184.954100,0.069181,46.737404,0.000000,0.091902,0.111592,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107545,CD14Mono:2026-06-27_16-48-26_4484adb6:45,10.122643,1.517220,373.246220,1096.802500,0.000000,40.097680,0.000000,0.493058,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107546,CD14Mono:2026-06-27_16-48-26_4484adb6:46,10.823056,2.675481,721.396670,0.478380,1.125787,3.693521,0.000000,0.000000,0.190955,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107547,CD14Mono:2026-06-27_16-48-26_4484adb6:47,11.561108,1.715770,734.997560,897.702400,0.517068,25.869467,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107548,CD14Mono:2026-06-27_16-48-26_4484adb6:48,10.204626,2.172140,1156.524000,0.000000,1.804664,55.672585,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Filter Optimization Data on Experimental Constraints.

### Define Function to Filter Optimization Data Based on Experimental Constraints.

In [9]:
def filter_opt_results(candidates_df, lbound_dict, ubound_dict,  margin = 0.1):
    # ----  Target Markers Loop ----
    print("---- Filtering Results ----")
    # define mask for all columns
    marker_mask = np.full(candidates_df.shape[0], True)

    # ----  Protocol Columns Loop ----
    for col, col_ubound in ubound_dict.items():
        # get col bounds
        col_lbound = lbound_dict[col]
        print(f"\t\t---- Processing Column {col} -> U: {col_ubound}, L: {col_lbound} ----")

        # get col values and mask
        col_values = candidates_df[col]
        col_mask_ubound_mask = col_values <= col_ubound*(1 + margin)
        col_mask_lbound_mask = col_values >= col_lbound*(1 - margin)

        # get masks for current column
        old_mask = marker_mask
        lbound_marker_mask = marker_mask & col_mask_lbound_mask
        ubound_marker_mask = marker_mask & col_mask_ubound_mask
        marker_mask = marker_mask & col_mask_ubound_mask & col_mask_lbound_mask

        # get number of solutions
        n_previously_accepted_solutions = old_mask.sum()
        n_accepted_solutions = marker_mask.sum()
        n_solutions_passing_ubound = col_mask_ubound_mask.sum()
        n_solutions_passing_lbound = col_mask_lbound_mask.sum()
        n_solutions_passing_ubound_and_previous_constraints = ubound_marker_mask.sum()
        n_solutions_passing_lbound_and_previous_constraints = lbound_marker_mask.sum()
        print(f"\t\t\t --- Number Previously Accepted Solutions {n_previously_accepted_solutions} ---")
        print(f"\t\t\t --- Number Currently Accepted Solutions {n_accepted_solutions} ---")
        print(f"\t\t\t --- Number of Solution Satisfying Upper Bound {n_solutions_passing_ubound} ---")
        print(f"\t\t\t --- Number of Solution Satisfying Lower Bound {n_solutions_passing_lbound} ---")
        print(f"\t\t\t --- Number of Solution Satisfying Upper Bound and Previous Constraints {n_solutions_passing_ubound_and_previous_constraints} ---")
        print(f"\t\t\t --- Number of Solution Satisfying Lower Bound and Previous Constraints {n_solutions_passing_lbound_and_previous_constraints} ---")


        # write values for constraint on current column
        candidates_df[f"passes_constraints:{col}:ubound"] = col_mask_ubound_mask
        candidates_df[f"passes_constraints:{col}:lbound"] = col_mask_ubound_mask

    # write values for all constraints
    n_all_solutions = candidates_df.shape[0]
    n_accepted_solutions = marker_mask.sum()
    candidates_df["passes_constraints:all"] = marker_mask

    print(f"\t\t\t--- Finished, Out of {n_all_solutions} there are {n_accepted_solutions} Passing with Margin {margin} ----")
    return candidates_df

### Filter Optimization Data from Loop 3.

In [11]:
opt_df3 = filter_opt_results(opt_df3, lbound3_dict, ubound3_dict)
opt_df3_filt = opt_df3[opt_df3["passes_constraints:all"]]
opt_df3_filt.shape[0] / opt_df3.shape[0]

---- Filtering Results ----
		---- Processing Column gm-csf_[ng_ml] -> U: 10.0, L: 0.0 ----
			 --- Number Previously Accepted Solutions 107550 ---
			 --- Number Currently Accepted Solutions 92712 ---
			 --- Number of Solution Satisfying Upper Bound 92712 ---
			 --- Number of Solution Satisfying Lower Bound 107550 ---
			 --- Number of Solution Satisfying Upper Bound and Previous Constraints 92712 ---
			 --- Number of Solution Satisfying Lower Bound and Previous Constraints 107550 ---
		---- Processing Column tpo_[ng_ml] -> U: 2.5, L: 0.0 ----
			 --- Number Previously Accepted Solutions 92712 ---
			 --- Number Currently Accepted Solutions 86106 ---
			 --- Number of Solution Satisfying Upper Bound 98052 ---
			 --- Number of Solution Satisfying Lower Bound 107550 ---
			 --- Number of Solution Satisfying Upper Bound and Previous Constraints 86106 ---
			 --- Number of Solution Satisfying Lower Bound and Previous Constraints 92712 ---
		---- Processing Column um171_[nm] -> U: 1120

/tmp/ipykernel_1664353/4143731902.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  candidates_df[f"passes_constraints:{col}:lbound"] = col_mask_ubound_mask
/tmp/ipykernel_1664353/4143731902.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  candidates_df[f"passes_constraints:{col}:ubound"] = col_mask_ubound_mask
/tmp/ipykernel_1664353/4143731902.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all colum

0.4303300790330079

## Get Optimization Data for Monocytes Targets.

### Subset the Optimization Data for the Target Cell Types.

In [55]:
ct_dfs_dict = {}
target_cts = ["CD14Mono", "CD16Mono"]

for ct in target_cts:
    ct_df3 = opt_df3_filt[opt_df3_filt["target_ct"] == ct]
    
    ct_dfs_dict[ct] = ct_df3

### Filter by Target Fate Enrichment.

In [58]:
base_dir = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/loop3_replicate"
percentile = 99

selected_runs_dict = {}

for ct, loop3_df in ct_dfs_dict.items():
    ct_enrichment_col = f"{ct}_prop"

    ct_props3 = loop3_df[ct_enrichment_col].values
    perc3 = np.percentile(ct_props3, percentile).item()
    mask3 = ct_props3 > perc3; n3 = mask3.sum(); tot3 = len(mask3)
    filt_loop3_df = loop3_df[mask3].copy()

    run_dir = filt_loop3_df.apply(lambda e: os.path.join(base_dir, e["opt_id"], e["target_ct"], e["run_id"]), axis=1)
    run_dir_unique = run_dir.unique(); nruns = len(run_dir_unique)
    print(f"Threshold -> {ct} || Percentile {perc3:f} || Passing {n3}/{tot3} || Unique runs {nruns}")

    selected_runs_dict[ct] = run_dir_unique.tolist()


Threshold -> CD14Mono || Percentile 0.141673 || Passing 149/14852 || Unique runs 82
Threshold -> CD16Mono || Percentile 0.297796 || Passing 97/9684 || Unique runs 37


### Save Runs to Run Evaluation for on Disk.

In [69]:
base_dump_dir = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/miscellaneous_new"
masked_eval_dump_dir = os.path.join(base_dump_dir, "masked_eval")
yaml_path = os.path.join(masked_eval_dump_dir, "runs_masked_eval.yaml")
os.makedirs(masked_eval_dump_dir, exist_ok=True)

# ---- Joint YAML file for both cell types ----
with open(yaml_path, "w") as fb:
    yaml.dump(selected_runs_dict, fb)


# ---- Per cell type YAML file ----
for ct, runs_list in selected_runs_dict.items():
    ct_yaml_path = os.path.join(masked_eval_dump_dir, f"{ct}.yaml")

    with open(ct_yaml_path, "w") as fb:
        yaml.dump(runs_list, fb)
    
# ---- Per cell type TXT file ----
for ct, runs_list in selected_runs_dict.items():
    ct_yaml_path = os.path.join(masked_eval_dump_dir, f"{ct}.txt")

    with open(ct_yaml_path, "w") as fb:
        for idx, run_id in enumerate(runs_list):
            # ---- Extra line only for intermediate runs ----
            fb.write(f"{run_id}\n")